# Notebook 6: Econometric Modelling

**Research Project:** Improving Asymmetric Exchange Rate Pass-Through Modelling Across Food Price Categories in South Africa Using Machine Learning

**Primary Modelling Period:** April 2017 – December 2025

**Unit of Analysis:** Food subclass × month

## Notebook Overview

This notebook develops the econometric component of the research using the
balanced food-subclass panel prepared in Notebook 5.

The econometric analysis has two purposes:

1. estimate a symmetric exchange-rate pass-through benchmark using ARDL models;
2. estimate asymmetric pass-through using nonlinear ARDL models that separate Rand depreciation and appreciation movements.

Models are estimated separately for each eligible food subclass. This allows the magnitude, direction and timing of exchange-rate pass-through to diffe across food categories.

The notebook covers model eligibility testing, stationarity analysis, lag selection and model estimation. Detailed diagnostics and research
interpretation will be completed in Notebook 7.

In [ ]:
# import required libraries
from pathlib import Path

import pandas as pd
import numpy as np
import warnings

from statsmodels.tsa.ardl import ARDL, UECM
from statsmodels.tsa.stattools import adfuller, kpss, zivot_andrews
from statsmodels.tools.sm_exceptions import InterpolationWarning

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)


In [17]:
# load the econometric dataset
data_path = Path("../data/processed/econometric_model_data.csv")

if not data_path.exists():
    raise FileNotFoundError(
        f"Econometric dataset not found: {data_path.resolve()}"
    )

econometric_data = pd.read_csv(
    data_path,
    parse_dates=["Date"]
)

econometric_data = (
    econometric_data
    .sort_values(["SubclassDescription", "Date"])
    .reset_index(drop=True)
)

print(f"Dataset shape: {econometric_data.shape}")
print(
    "Period:",
    econometric_data["Date"].min().date(),
    "to",
    econometric_data["Date"].max().date()
)
print(
    "Food subclasses:",
    econometric_data["SubclassDescription"].nunique()
)

Dataset shape: (4830, 15)
Period: 2017-04-01 to 2025-12-01
Food subclasses: 46


In [18]:
# validate the modelling structure
required_columns = [
    "Date",
    "GroupDescription",
    "ClassDescription",
    "SubclassDescription",
    "Subclass_Weight",
    "CPI",
    "Log_CPI",
    "Food_Inflation_Pct",
    "ExchangeRate",
    "Log_ExchangeRate",
    "ExchangeRate_Log_Change_Pct",
    "Depreciation_Shock_Pct",
    "Appreciation_Shock_Pct",
    "ExchangeRate_Positive_Cumulative_Pct",
    "ExchangeRate_Negative_Cumulative_Pct"
]

missing_columns = sorted(
    set(required_columns) - set(econometric_data.columns)
)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

duplicate_count = econometric_data.duplicated(
    subset=["Date", "SubclassDescription"]
).sum()

missing_value_count = (
    econometric_data[required_columns]
    .isna()
    .sum()
    .sum()
)

subclass_month_counts = (
    econometric_data
    .groupby("SubclassDescription")["Date"]
    .nunique()
)

exchange_rate_columns = [
    "ExchangeRate",
    "Log_ExchangeRate",
    "ExchangeRate_Log_Change_Pct",
    "ExchangeRate_Positive_Cumulative_Pct",
    "ExchangeRate_Negative_Cumulative_Pct"
]

maximum_monthly_fx_values = (
    econometric_data
    .groupby("Date")[exchange_rate_columns]
    .nunique()
    .max()
    .max()
)

if duplicate_count != 0:
    raise ValueError("Duplicate subclass-month observations detected.")

if missing_value_count != 0:
    raise ValueError("Missing values detected in required variables.")

if subclass_month_counts.nunique() != 1:
    raise ValueError("Food subclasses do not have equal time coverage.")

if maximum_monthly_fx_values != 1:
    raise ValueError("Inconsistent exchange-rate values detected within months.")

validation_summary = pd.Series({
    "Observations": len(econometric_data),
    "Food subclasses": econometric_data[
        "SubclassDescription"
    ].nunique(),
    "Unique months": econometric_data["Date"].nunique(),
    "Minimum months per subclass": subclass_month_counts.min(),
    "Maximum months per subclass": subclass_month_counts.max(),
    "Duplicate subclass-month rows": duplicate_count,
    "Missing required values": missing_value_count,
    "Maximum FX values within a month": maximum_monthly_fx_values
})

validation_summary.to_frame(name="Value")

,Value
Observations,4830
Food subclasses,46
Unique months,105
Minimum months per subclass,105
Maximum months per subclass,105
Duplicate subclass-month rows,0
Missing required values,0
Maximum FX values within a month,1


### Modelling-Structure Validation

The validation confirms that the econometric dataset contains 4,830
observations representing 46 food subclasses across 105 common months.

Every subclass has complete coverage from April 2017 to December 2025. No duplicate subclass-month observations or missing modelling values were identified.

The exchange-rate variables are also consistent within each month, confirming that all food subclasses are matched to the same monthly macroeconomic series.

The dataset therefore satisfies the structural requirements for
subclass-specific time-series estimation.

### Econometric Modelling Framework

Two related econometric specifications will be estimated for each food
subclass.

### Symmetric ARDL Benchmark

The symmetric ARDL model uses log food CPI as the dependent variable and the log USD/ZAR exchange rate as the principal explanatory variable.

This specification assumes that Rand depreciation and appreciation have equal but opposite effects on food prices. It provides the conventional benchmark against which the asymmetric model can be assessed.

### Asymmetric NARDL Model

The NARDL specification replaces the single exchange-rate variable with its cumulative positive and negative components:

- `ExchangeRate_Positive_Cumulative_Pct` represents cumulative Rand
  depreciation.
- `ExchangeRate_Negative_Cumulative_Pct` represents cumulative Rand
  appreciation and remains negatively signed.

This decomposition allows depreciation and appreciation to have different short-run and long-run relationships with food prices.

### Dependent Variable

`Log_CPI` is used as the level-form dependent variable in the ARDL and NARDL specifications. Its first difference represents monthly food-price inflation.

Before estimating either model, the integration order of each variable must be assessed. ARDL bounds-testing methods permit a mixture of I(0) and I(1) variables but are not valid when any model variable is integrated of order two, I(2).

In [19]:
# stationarity-testing function
def run_stationarity_tests(
    series,
    variable_name,
    significance_level=0.05
):
    clean_series = (
        pd.Series(series)
        .dropna()
        .astype(float)
    )

    transformations = {
        "Level": clean_series,
        "First difference": clean_series.diff().dropna()
    }

    test_results = []

    for transformation, transformed_series in transformations.items():
        adf_result = adfuller(
            transformed_series,
            regression="c",
            autolag="AIC"
        )

        with warnings.catch_warnings():
            warnings.simplefilter(
                "ignore",
                category=InterpolationWarning
            )

            kpss_result = kpss(
                transformed_series,
                regression="c",
                nlags="auto"
            )

        test_results.append({
            "Variable": variable_name,
            "Transformation": transformation,
            "Observations": len(transformed_series),
            "ADF_Statistic": adf_result[0],
            "ADF_P_Value": adf_result[1],
            "ADF_Lags": adf_result[2],
            "ADF_Stationary": (
                adf_result[1] < significance_level
            ),
            "KPSS_Statistic": kpss_result[0],
            "KPSS_P_Value": kpss_result[1],
            "KPSS_Lags": kpss_result[2],
            "KPSS_Stationary": (
                kpss_result[1] >= significance_level
            )
        })

    return pd.DataFrame(test_results)

In [20]:
# prepare the unique monthly exchange-rate series
monthly_econometric_data = (
    econometric_data[
        [
            "Date",
            "Log_ExchangeRate",
            "ExchangeRate_Log_Change_Pct",
            "ExchangeRate_Positive_Cumulative_Pct",
            "ExchangeRate_Negative_Cumulative_Pct"
        ]
    ]
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .reset_index(drop=True)
)

exchange_rate_test_variables = [
    "Log_ExchangeRate",
    "ExchangeRate_Log_Change_Pct",
    "ExchangeRate_Positive_Cumulative_Pct",
    "ExchangeRate_Negative_Cumulative_Pct"
]

exchange_rate_stationarity_results = pd.concat(
    [
        run_stationarity_tests(
            monthly_econometric_data[variable],
            variable
        )
        for variable in exchange_rate_test_variables
    ],
    ignore_index=True
)

exchange_rate_stationarity_results

,Variable,Transformation,Observations,ADF_Statistic,ADF_P_Value,ADF_Lags,ADF_Stationary,KPSS_Statistic,KPSS_P_Value,KPSS_Lags,KPSS_Stationary
0,Log_ExchangeRate,Level,105,-1.864525,0.348929,1,False,1.280760,0.010000,6,False
1,Log_ExchangeRate,First difference,104,-8.134200,0.000000,0,True,0.071541,0.100000,1,True
2,ExchangeRate_Log_Change_Pct,Level,105,-8.288868,0.000000,0,True,0.080543,0.100000,1,True
3,ExchangeRate_Log_Change_Pct,First difference,104,-6.269094,0.000000,9,True,0.157573,0.100000,30,True
4,ExchangeRate_Positive_Cumulative_Pct,Level,105,-1.683814,0.439471,1,False,1.578454,0.010000,6,False
5,ExchangeRate_Positive_Cumulative_Pct,First difference,104,-8.252650,0.000000,0,True,0.293281,0.100000,2,True
6,ExchangeRate_Negative_Cumulative_Pct,Level,105,-1.505169,0.530953,0,False,1.586885,0.010000,6,False
7,ExchangeRate_Negative_Cumulative_Pct,First difference,104,-7.863987,0.000000,0,True,0.199321,0.100000,1,True


In [21]:
# test log CPI separately for each food subclass
food_price_stationarity_results = []

for subclass_name, subclass_data in econometric_data.groupby(
    "SubclassDescription"
):
    subclass_results = run_stationarity_tests(
        subclass_data["Log_CPI"],
        "Log_CPI"
    )

    subclass_results.insert(
        0,
        "SubclassDescription",
        subclass_name
    )

    food_price_stationarity_results.append(subclass_results)

food_price_stationarity_results = pd.concat(
    food_price_stationarity_results,
    ignore_index=True
)

food_price_stationarity_results.head(10)

,SubclassDescription,Variable,Transformation,Observations,ADF_Statistic,ADF_P_Value,ADF_Lags,ADF_Stationary,KPSS_Statistic,KPSS_P_Value,KPSS_Lags,KPSS_Stationary
0,Baby food,Log_CPI,Level,105,-1.216722,0.666361,7,False,1.486630,0.010000,6,False
1,Baby food,Log_CPI,First difference,104,-1.751753,0.404664,6,False,0.370346,0.089937,2,True
2,Bread and bakery products,Log_CPI,Level,105,-0.203780,0.938070,1,False,1.565412,0.010000,6,False
3,Bread and bakery products,Log_CPI,First difference,104,-6.827378,0.000000,0,True,0.149196,0.100000,4,True
4,Breakfast cereals,Log_CPI,Level,105,-0.678768,0.852173,0,False,1.551419,0.010000,6,False
5,Breakfast cereals,Log_CPI,First difference,104,-10.709926,0.000000,0,True,0.180325,0.100000,1,True
6,Cereals,Log_CPI,Level,105,-1.183943,0.680465,5,False,1.403764,0.010000,6,False
7,Cereals,Log_CPI,First difference,104,-4.254948,0.000531,2,True,0.202674,0.100000,2,True
8,Cheese,Log_CPI,Level,105,0.557774,0.986527,0,False,1.563846,0.010000,6,False
9,Cheese,Log_CPI,First difference,104,-10.890011,0.000000,0,True,0.209836,0.100000,0,True


In [22]:
# summarise stationarity decisions across food subclasses
food_price_stationarity_summary = (
    food_price_stationarity_results
    .groupby("Transformation")
    .agg(
        Food_Subclasses=(
            "SubclassDescription",
            "nunique"
        ),
        ADF_Stationary=(
            "ADF_Stationary",
            "sum"
        ),
        KPSS_Stationary=(
            "KPSS_Stationary",
            "sum"
        )
    )
    .reindex(["Level", "First difference"])
)

food_price_stationarity_summary

,Food_Subclasses,ADF_Stationary,KPSS_Stationary
Transformation,,,
Level,46,0,1
First difference,46,42,44


### Initial Stationarity Interpretation

The log exchange rate is non-stationary in levels but stationary after first differencing according to both the ADF and KPSS tests. It is therefore classified as I(1).

The cumulative depreciation and appreciation components follow the same
pattern. Both are non-stationary in levels and stationary after first
differencing, supporting their inclusion as I(1) variables in the NARDL
framework.

The monthly log exchange-rate change is stationary in levels and is therefore classified as I(0).

For food prices, none of the 46 subclass log CPI series was stationary in levels according to the ADF test, while only one was classified as stationary by the KPSS test. After first differencing, the ADF test classified 42 subclasses as stationary and the KPSS test classified 44 as stationary.

Most food-price series therefore appear to be I(1). However, subclasses for which the tests disagree or fail to establish first-difference stationarity must be examined before model estimation.

In [23]:
# classify food-price integration orders
level_test_results = (
    food_price_stationarity_results[
        food_price_stationarity_results[
            "Transformation"
        ] == "Level"
    ]
    .set_index("SubclassDescription")
    [
        [
            "ADF_P_Value",
            "ADF_Stationary",
            "KPSS_P_Value",
            "KPSS_Stationary"
        ]
    ]
    .rename(
        columns={
            "ADF_P_Value": "Level_ADF_P_Value",
            "ADF_Stationary": "Level_ADF_Stationary",
            "KPSS_P_Value": "Level_KPSS_P_Value",
            "KPSS_Stationary": "Level_KPSS_Stationary"
        }
    )
)

difference_test_results = (
    food_price_stationarity_results[
        food_price_stationarity_results[
            "Transformation"
        ] == "First difference"
    ]
    .set_index("SubclassDescription")
    [
        [
            "ADF_P_Value",
            "ADF_Stationary",
            "KPSS_P_Value",
            "KPSS_Stationary"
        ]
    ]
    .rename(
        columns={
            "ADF_P_Value": "Difference_ADF_P_Value",
            "ADF_Stationary": "Difference_ADF_Stationary",
            "KPSS_P_Value": "Difference_KPSS_P_Value",
            "KPSS_Stationary": "Difference_KPSS_Stationary"
        }
    )
)

food_price_integration_status = level_test_results.join(
    difference_test_results
)

level_stationary = (
    food_price_integration_status["Level_ADF_Stationary"]
    & food_price_integration_status["Level_KPSS_Stationary"]
)

difference_stationary = (
    food_price_integration_status["Difference_ADF_Stationary"]
    & food_price_integration_status["Difference_KPSS_Stationary"]
)

food_price_integration_status["Integration_Order"] = np.select(
    [
        level_stationary,
        ~level_stationary & difference_stationary
    ],
    [
        "I(0)",
        "I(1)"
    ],
    default="Requires review"
)

food_price_integration_status[
    "Integration_Order"
].value_counts().to_frame(name="Food_Subclasses")

,Food_Subclasses
Integration_Order,
I(1),40
Requires review,6


### Review of Inconclusive Food-Price Series

A conservative classification requires both tests to support stationarity.

Subclasses are classified as I(0) when both tests indicate stationarity in levels and as I(1) when both tests indicate stationarity after first differencing.

A `Requires review` result does not automatically mean that the series is I(2). It indicates that the two tests disagree or that first-difference stationarity has not yet been established conclusively.

In [24]:
# inspect subclasses with inconclusive results
food_price_series_for_review = (
    food_price_integration_status[
        food_price_integration_status[
            "Integration_Order"
        ] == "Requires review"
    ]
    .sort_values(
        [
            "Difference_ADF_P_Value",
            "Difference_KPSS_P_Value"
        ],
        ascending=False
    )
)

food_price_series_for_review

,Level_ADF_P_Value,Level_ADF_Stationary,Level_KPSS_P_Value,Level_KPSS_Stationary,Difference_ADF_P_Value,Difference_ADF_Stationary,Difference_KPSS_P_Value,Difference_KPSS_Stationary,Integration_Order
SubclassDescription,,,,,,,,,
"Stone fruits and pome fruits, fresh",0.943496,False,0.010000,False,0.525855,False,0.100000,True,Requires review
Baby food,0.666361,False,0.010000,False,0.404664,False,0.089937,True,Requires review
"Salt, condiments and sauces",0.905076,False,0.010000,False,0.348858,False,0.100000,True,Requires review
Other food products n.e.c.,0.858553,False,0.010000,False,0.152017,False,0.100000,True,Requires review
Coffee and coffee substitutes,0.998312,False,0.010000,False,0.001276,True,0.010000,False,Requires review
"Chocolate, cocoa, and cocoa-based food products",0.998883,False,0.010000,False,0.000000,True,0.010000,False,Requires review


### Robustness Tests for Inconclusive Series

Six food subclasses could not be classified using the initial joint decision rule.

For four subclasses, the KPSS test supports first-difference stationarity but the ADF test does not reject a unit root. For coffee and chocolate products, the ADF test supports stationarity but the KPSS test does not.

These series are reassessed using alternative ADF lag-selection rules and different KPSS bandwidths. This determines whether the conclusions are sensitive to a particular test specification.

The robustness checks are applied only to the six unresolved subclasses. They are not used to select whichever individual result is most favourable.

In [25]:
# test ADF sensitivity to lag selection
review_subclasses = (
    food_price_series_for_review
    .index
    .tolist()
)

adf_specifications = {
    "AIC": {
        "maxlag": 12,
        "autolag": "AIC"
    },
    "BIC": {
        "maxlag": 12,
        "autolag": "BIC"
    },
    "Fixed lag 0": {
        "maxlag": 0,
        "autolag": None
    },
    "Fixed lag 1": {
        "maxlag": 1,
        "autolag": None
    },
    "Fixed lag 3": {
        "maxlag": 3,
        "autolag": None
    },
    "Fixed lag 6": {
        "maxlag": 6,
        "autolag": None
    }
}

adf_robustness_records = []

for subclass_name in review_subclasses:
    subclass_series = (
        econometric_data[
            econometric_data["SubclassDescription"] == subclass_name
        ]
        .sort_values("Date")["Log_CPI"]
        .diff()
        .dropna()
    )

    for specification, settings in adf_specifications.items():
        test_result = adfuller(
            subclass_series,
            regression="c",
            **settings
        )

        adf_robustness_records.append({
            "SubclassDescription": subclass_name,
            "Specification": specification,
            "ADF_Statistic": test_result[0],
            "ADF_P_Value": test_result[1],
            "Selected_Lags": test_result[2],
            "Stationary": test_result[1] < 0.05
        })

adf_robustness_results = pd.DataFrame(
    adf_robustness_records
)

In [26]:
# display ADF p-values across specifications
adf_robustness_p_values = (
    adf_robustness_results
    .pivot(
        index="SubclassDescription",
        columns="Specification",
        values="ADF_P_Value"
    )
    .reindex(columns=adf_specifications.keys())
)

adf_robustness_p_values

Specification,AIC,BIC,Fixed lag 0,Fixed lag 1,Fixed lag 3,Fixed lag 6
SubclassDescription,,,,,,
Baby food,0.404664,0.000000,0.000000,0.000000,0.000172,0.404664
"Chocolate, cocoa, and cocoa-based food products",0.339139,0.000000,0.000000,0.000000,0.000164,0.079142
Coffee and coffee substitutes,0.001276,0.000000,0.000000,0.000000,0.002514,0.093880
Other food products n.e.c.,0.242231,0.000000,0.000000,0.000000,0.000527,0.242231
"Salt, condiments and sauces",0.348858,0.000000,0.000000,0.000000,0.000012,0.009600
"Stone fruits and pome fruits, fresh",0.525855,0.309753,0.000000,0.000000,0.000003,0.000001


In [27]:
# display lags used in each ADF specification
adf_selected_lags = (
    adf_robustness_results
    .pivot(
        index="SubclassDescription",
        columns="Specification",
        values="Selected_Lags"
    )
    .reindex(columns=adf_specifications.keys())
)

adf_selected_lags

Specification,AIC,BIC,Fixed lag 0,Fixed lag 1,Fixed lag 3,Fixed lag 6
SubclassDescription,,,,,,
Baby food,6,0,0,1,3,6
"Chocolate, cocoa, and cocoa-based food products",8,0,0,1,3,6
Coffee and coffee substitutes,2,0,0,1,3,6
Other food products n.e.c.,6,0,0,1,3,6
"Salt, condiments and sauces",12,0,0,1,3,6
"Stone fruits and pome fruits, fresh",12,11,0,1,3,6


In [28]:
# test KPSS sensitivity to bandwidth selection
kpss_specifications = {
    "Automatic": "auto",
    "Lag 1": 1,
    "Lag 3": 3,
    "Lag 6": 6,
    "Lag 12": 12
}

kpss_robustness_records = []

for subclass_name in review_subclasses:
    subclass_series = (
        econometric_data[
            econometric_data["SubclassDescription"] == subclass_name
        ]
        .sort_values("Date")["Log_CPI"]
        .diff()
        .dropna()
    )

    for specification, lag_setting in kpss_specifications.items():
        with warnings.catch_warnings():
            warnings.simplefilter(
                "ignore",
                category=InterpolationWarning
            )

            test_result = kpss(
                subclass_series,
                regression="c",
                nlags=lag_setting
            )

        kpss_robustness_records.append({
            "SubclassDescription": subclass_name,
            "Specification": specification,
            "KPSS_Statistic": test_result[0],
            "KPSS_P_Value": test_result[1],
            "Selected_Lags": test_result[2],
            "Stationary": test_result[1] >= 0.05
        })

kpss_robustness_results = pd.DataFrame(
    kpss_robustness_records
)

In [29]:
# display KPSS p-values across bandwidths
kpss_robustness_p_values = (
    kpss_robustness_results
    .pivot(
        index="SubclassDescription",
        columns="Specification",
        values="KPSS_P_Value"
    )
    .reindex(columns=kpss_specifications.keys())
)

kpss_robustness_p_values

Specification,Automatic,Lag 1,Lag 3,Lag 6,Lag 12
SubclassDescription,,,,,
Baby food,0.089937,0.076703,0.094950,0.100000,0.100000
"Chocolate, cocoa, and cocoa-based food products",0.010000,0.010000,0.010000,0.010000,0.040656
Coffee and coffee substitutes,0.010000,0.010000,0.010000,0.010000,0.021132
Other food products n.e.c.,0.100000,0.100000,0.100000,0.100000,0.100000
"Salt, condiments and sauces",0.100000,0.100000,0.100000,0.100000,0.100000
"Stone fruits and pome fruits, fresh",0.100000,0.100000,0.100000,0.100000,0.100000


### Robustness-Test Interpretation

The robustness tests indicate that the initial ADF failures for baby food, other food products, salt, condiments and sauces, and stone fruits are sensitive to the number of lags included in the test.

For baby food and other food products, the AIC specification selected six lags and failed to reject a unit root. However, BIC selected zero lags, and the fixed zero-, one- and three-lag specifications strongly supported first-difference stationarity. The KPSS test also supported stationarity across all bandwidths.

Salt, condiments and sauces followed a similar pattern. AIC selected 12 lags and failed to reject a unit root, while BIC and the fixed-lag specifications supported stationarity.

Stone fruits and pome fruits remained sensitive under the high-lag ADF
specifications. However, the fixed zero-, one-, three- and six-lag tests rejected a unit root, while KPSS consistently supported stationarity. The series is therefore consistent with I(1), although its classification is lag-sensitive.

Coffee and chocolate products require additional investigation. Their ADF results generally support first-difference stationarity, but KPSS rejects stationarity across all tested bandwidths. A structural break may account for this disagreement.

### Structural-Break Stationarity Test

The Zivot-Andrews test is used for the two subclasses with persistent
ADF–KPSS disagreement.

Unlike the standard ADF test, the Zivot-Andrews test allows for one
endogenously identified structural break. Its null hypothesis is that the series contains a unit root, including when a possible structural break is considered.

A p-value below 0.05 supports stationarity around a structural break and provides evidence against the series being I(2).

In [30]:
# test first differences with one structural break
structural_break_subclasses = [
    "Coffee and coffee substitutes",
    "Chocolate, cocoa, and cocoa-based food products"
]

structural_break_records = []

for subclass_name in structural_break_subclasses:
    subclass_series = (
        econometric_data[
            econometric_data["SubclassDescription"] == subclass_name
        ]
        .sort_values("Date")
        .set_index("Date")["Log_CPI"]
        .diff()
        .dropna()
    )

    test_result = zivot_andrews(
        subclass_series,
        trim=0.15,
        maxlag=12,
        regression="c",
        autolag="BIC"
    )

    break_position = test_result[4]
    break_date = subclass_series.index[break_position]

    structural_break_records.append({
        "SubclassDescription": subclass_name,
        "ZA_Statistic": test_result[0],
        "ZA_P_Value": test_result[1],
        "Selected_Lags": test_result[3],
        "Break_Date": break_date,
        "Stationary_With_Break": test_result[1] < 0.05
    })

structural_break_results = pd.DataFrame(
    structural_break_records
).set_index("SubclassDescription")

structural_break_results

,ZA_Statistic,ZA_P_Value,Selected_Lags,Break_Date,Stationary_With_Break
SubclassDescription,,,,,
Coffee and coffee substitutes,-12.992023,0.000010,0,2022-01-01,True
"Chocolate, cocoa, and cocoa-based food products",-11.477148,0.000010,0,2024-03-01,True


### Structural-Break Test Interpretation

The Zivot-Andrews test strongly rejects the unit-root null for the first differences of both unresolved food-price series.

Coffee and coffee substitutes recorded a structural-break date of January 2022, while chocolate, cocoa and cocoa-based food products recorded a break in March 2024.

These results indicate that the first differences are stationary when a
possible structural break is considered. The earlier ADF–KPSS disagreement therefore does not provide evidence that either series is I(2).

Both subclasses are retained in the econometric sample and classified as I(1),with their structural-break sensitivity documented for subsequent model interpretation.

In [31]:
# finalise food-price integration classifications
lag_sensitive_subclasses = [
    "Baby food",
    "Other food products n.e.c.",
    "Salt, condiments and sauces",
    "Stone fruits and pome fruits, fresh"
]

break_adjusted_subclasses = [
    "Coffee and coffee substitutes",
    "Chocolate, cocoa, and cocoa-based food products"
]

food_price_integration_status["Classification_Basis"] = np.where(
    food_price_integration_status["Integration_Order"] == "I(1)",
    "ADF and KPSS agree",
    pd.NA
)

food_price_integration_status.loc[
    lag_sensitive_subclasses,
    ["Integration_Order", "Classification_Basis"]
] = [
    "I(1)",
    "Stationary after differencing; ADF lag-sensitive"
]

food_price_integration_status.loc[
    break_adjusted_subclasses,
    ["Integration_Order", "Classification_Basis"]
] = [
    "I(1)",
    "Stationary after differencing with structural break"
]

unresolved_series = food_price_integration_status[
    ~food_price_integration_status["Integration_Order"].isin(
        ["I(0)", "I(1)"]
    )
]

if not unresolved_series.empty:
    raise ValueError(
        "Some food-price series remain unresolved."
    )

food_price_integration_status[
    ["Integration_Order", "Classification_Basis"]
].head(10)

,Integration_Order,Classification_Basis
SubclassDescription,,
Baby food,I(1),Stationary after differencing; ADF lag-sensitive
Bread and bakery products,I(1),ADF and KPSS agree
Breakfast cereals,I(1),ADF and KPSS agree
Cereals,I(1),ADF and KPSS agree
Cheese,I(1),ADF and KPSS agree
"Chocolate, cocoa, and cocoa-based food products",I(1),Stationary after differencing with structural ...
Coffee and coffee substitutes,I(1),Stationary after differencing with structural ...
"Dates, figs and tropical fruits, fresh",I(1),ADF and KPSS agree
Eggs,I(1),ADF and KPSS agree


In [32]:
# summarise integration orders for all model components
model_integration_summary = pd.DataFrame({
    "Model_Component": [
        "Food-price level",
        "Symmetric exchange-rate level",
        "Monthly exchange-rate change",
        "Cumulative depreciation component",
        "Cumulative appreciation component"
    ],
    "Variable": [
        "Log_CPI",
        "Log_ExchangeRate",
        "ExchangeRate_Log_Change_Pct",
        "ExchangeRate_Positive_Cumulative_Pct",
        "ExchangeRate_Negative_Cumulative_Pct"
    ],
    "Integration_Order": [
        "I(1)",
        "I(1)",
        "I(0)",
        "I(1)",
        "I(1)"
    ],
    "Scope": [
        "46 subclass series",
        "Common monthly series",
        "Common monthly series",
        "Common monthly series",
        "Common monthly series"
    ]
})

model_integration_summary

,Model_Component,Variable,Integration_Order,Scope
0,Food-price level,Log_CPI,I(1),46 subclass series
1,Symmetric exchange-rate level,Log_ExchangeRate,I(1),Common monthly series
2,Monthly exchange-rate change,ExchangeRate_Log_Change_Pct,I(0),Common monthly series
3,Cumulative depreciation component,ExchangeRate_Positive_Cumulative_Pct,I(1),Common monthly series
4,Cumulative appreciation component,ExchangeRate_Negative_Cumulative_Pct,I(1),Common monthly series


### Stationarity Conclusion

All 46 log food CPI series are classified as I(1). Forty subclasses received consistent ADF and KPSS support, four were classified after examining lag-selection sensitivity, and two were classified using a
structural-break-aware test.

The log exchange rate and its cumulative positive and negative components are also I(1), while the monthly log exchange-rate change is I(0).

No variable included in the proposed econometric specifications was found to be I(2). The integration-order results therefore permit ARDL and NARDL estimation.

These findings establish model eligibility but do not by themselves establish cointegration. The existence of a long-run relationship must be assessed using the bounds-testing procedure after model estimation.